In [4]:
import duckdb
import pandas as pd
from sklearn.model_selection import train_test_split

In [5]:
# Подключаемся к датасету через DuckDB
con = duckdb.connect(database=':memory:')
# ВАЖНО: путь может отличаться в зависимости от того, откуда запускается код
data_path = '../../chatbots_dataset_labeled.parquet'

# Загрузка полного датасета
df = con.execute(f"SELECT * FROM '{data_path}'").df()

# Статистика 
stats_query = f"""
SELECT 
    COUNT(*) as total_rows,
    SUM(CASE WHEN message_label = 'illegal' THEN 1 ELSE 0 END) as illegal_count,
    SUM(CASE WHEN message_label = 'legal' THEN 1 ELSE 0 END) as legal_count
FROM '{data_path}';
"""

stats = con.execute(stats_query).df()
print(f"Всего строк: {stats['total_rows'][0]:,}")
print(f"Illegal: {stats['illegal_count'][0]:,}")
print(f"Legal: {stats['legal_count'][0]:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Всего строк: 7,816,579
Illegal: 463.0
Legal: 7,816,116.0


In [6]:
# Распределение меток через DuckDB
label_dist_query = f"""
SELECT 
    message_label,
    COUNT(*) as count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as percentage
FROM '{data_path}'
GROUP BY message_label;
"""

label_dist = con.execute(label_dist_query).df()
print("Распределение message_label:")
display(label_dist)


Распределение message_label:


,message_label,count,percentage
0,illegal,463,0.01
1,legal,7816116,99.99


In [7]:
# 1. Находим сессии с illegal сообщениями
sessions_with_illegal_query = f"""
SELECT DISTINCT session_id
FROM '{data_path}'
WHERE message_label = 'illegal';
"""

illegal_sessions = con.execute(sessions_with_illegal_query).df()['session_id'].tolist()
print(f"Сессий с illegal контентом: {len(illegal_sessions)}")

# 2. Извлекаем ВСЕ сообщения из этих сессий
sessions_str = "', '".join(illegal_sessions)
illegal_sessions_data_query = f"""
SELECT * 
FROM '{data_path}'
WHERE session_id IN ('{sessions_str}');
"""

df_illegal_sessions = con.execute(illegal_sessions_data_query).df()
print(f"Всего сообщений из illegal сессий: {len(df_illegal_sessions)}")

# 3. Считаем, сколько "чистых" legal сессий нужно добавить
# Целевое соотношение сообщений: примерно 1:2 (illegal:legal)
num_illegal_messages = (df_illegal_sessions['message_label'] == 'illegal').sum()
num_legal_messages = (df_illegal_sessions['message_label'] == 'legal').sum()
target_legal = num_illegal_messages * 2
need_additional_legal = max(0, target_legal - num_legal_messages)

print(f"\nВ illegal сессиях:")
print(f"  illegal сообщений: {num_illegal_messages}")
print(f"  legal сообщений: {num_legal_messages}")
print(f"  Нужно добавить legal сообщений: {need_additional_legal}")

# 4. Берём случайные "чистые" сессии (где НЕТ illegal сообщений)
# Сначала находим такие сессии
clean_sessions_query = f"""
SELECT DISTINCT session_id
FROM '{data_path}'
WHERE session_id NOT IN ('{sessions_str}')
    AND message_label = 'legal';
"""

clean_sessions = con.execute(clean_sessions_query).df()['session_id'].tolist()
print(f"\nВсего чистых legal сессий: {len(clean_sessions)}")

# 5. Берём сообщения из случайных чистых сессий до нужного количества
# Делаем это итеративно через DuckDB
import random
random.seed(42)
random.shuffle(clean_sessions)

# Берём сессии, пока не наберём нужное количество сообщений
selected_clean_sessions = []
accumulated_messages = 0

for session_id in clean_sessions:
    if accumulated_messages >= need_additional_legal:
        break
    
    # Считаем сообщения в этой сессии
    count_query = f"""
    SELECT COUNT(*) as cnt
    FROM '{data_path}'
    WHERE session_id = '{session_id}';
    """
    session_count = con.execute(count_query).df()['cnt'][0]
    
    selected_clean_sessions.append(session_id)
    accumulated_messages += session_count

print(f"Выбрано чистых сессий: {len(selected_clean_sessions)}")
print(f"Они содержат примерно {accumulated_messages} сообщений")

# 6. Извлекаем сообщения из выбранных чистых сессий
if len(selected_clean_sessions) > 0:
    clean_sessions_str = "', '".join(selected_clean_sessions)
    clean_data_query = f"""
    SELECT * 
    FROM '{data_path}'
    WHERE session_id IN ('{clean_sessions_str}');
    """
    
    df_clean_sessions = con.execute(clean_data_query).df()
    print(f"Загружено сообщений из чистых сессий: {len(df_clean_sessions)}")
    
    # 7. Объединяем illegal сессии + чистые сессии
    df_balanced = pd.concat([df_illegal_sessions, df_clean_sessions], ignore_index=True)
else:
    df_balanced = df_illegal_sessions.copy()

# 8. Перемешиваем
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nСбалансированный датасет:")
print(f"Всего строк: {len(df_balanced)}")
print(f"Всего сессий: {df_balanced['session_id'].nunique()}")
print(f"\nРаспределение message_label:")
print(df_balanced['message_label'].value_counts())

# Закрываем DuckDB соединение
con.close()

Сессий с illegal контентом: 115
Всего сообщений из illegal сессий: 1156

В illegal сессиях:
  illegal сообщений: 463
  legal сообщений: 693
  Нужно добавить legal сообщений: 233

Всего чистых legal сессий: 3581002
Выбрано чистых сессий: 101
Они содержат примерно 236 сообщений
Загружено сообщений из чистых сессий: 236

Сбалансированный датасет:
Всего строк: 1392
Всего сессий: 216

Распределение message_label:
message_label
legal      929
illegal    463
Name: count, dtype: int64


In [8]:
# 1. Получаем список всех сессий
all_sessions = df_balanced['session_id'].unique()
print(f"Всего сессий: {len(all_sessions)}")

# 2. Разделяем сессии на train/val/test (70/15/15)
sessions_train_val, sessions_test = train_test_split(
    all_sessions,
    test_size=0.15,
    random_state=42
)

sessions_train, sessions_val = train_test_split(
    sessions_train_val,
    test_size=0.15 / 0.85,
    random_state=42
)

print(f"\nСессий в train: {len(sessions_train)}")
print(f"Сессий в val: {len(sessions_val)}")
print(f"Сессий в test: {len(sessions_test)}")

# 3. Фильтруем сообщения по сессиям
df_train = df_balanced[df_balanced['session_id'].isin(sessions_train)]
df_val = df_balanced[df_balanced['session_id'].isin(sessions_val)]
df_test = df_balanced[df_balanced['session_id'].isin(sessions_test)]

print("\nРазделение датасета:")
print(f"Train: {len(df_train)} сообщений из {len(sessions_train)} сессий")
print(f"  illegal: {(df_train['message_label']=='illegal').sum()}")
print(f"  legal: {(df_train['message_label']=='legal').sum()}")

print(f"\nValidation: {len(df_val)} сообщений из {len(sessions_val)} сессий")
print(f"  illegal: {(df_val['message_label']=='illegal').sum()}")
print(f"  legal: {(df_val['message_label']=='legal').sum()}")

print(f"\nTest: {len(df_test)} сообщений из {len(sessions_test)} сессий")
print(f"  illegal: {(df_test['message_label']=='illegal').sum()}")
print(f"  legal: {(df_test['message_label']=='legal').sum()}")

# 4. Проверяем, что сессии не пересекаются
train_sessions = set(df_train['session_id'])
val_sessions = set(df_val['session_id'])
test_sessions = set(df_test['session_id'])

assert len(train_sessions & val_sessions) == 0, "Сессии пересекаются между train и val!"
assert len(train_sessions & test_sessions) == 0, "Сессии пересекаются между train и test!"
assert len(val_sessions & test_sessions) == 0, "Сессии пересекаются между val и test!"

print("\nПроверка пройдена: сессии не пересекаются между датасетами")

Всего сессий: 216

Сессий в train: 150
Сессий в val: 33
Сессий в test: 33

Разделение датасета:
Train: 1000 сообщений из 150 сессий
  illegal: 363
  legal: 637

Validation: 236 сообщений из 33 сессий
  illegal: 64
  legal: 172

Test: 156 сообщений из 33 сессий
  illegal: 36
  legal: 120

Проверка пройдена: сессии не пересекаются между датасетами


In [ ]:
# Сохраняем в parquet (эффективнее для больших данных, чем CSV)
df_train.to_parquet('../data/processed/train.parquet', index=False)
df_val.to_parquet('../data/processed/val.parquet', index=False)
df_test.to_parquet('../data/processed/test.parquet', index=False)

print("Датасеты сохранены в data/processed/")